Дицель Н.С. ИКБО-11-23

Информационные системы мониторинга и управления инцидентами

# Практическая работа 7. Open-Source IDS Snort

Snort 3.13.2.0 и DAQ 3.0.29 собраны из исходных кодов в Ubuntu 24.04 (WSL2).
Для повторного запуска необходимы Ubuntu/WSL Ubuntu, доступ в Интернет и права root/sudo.

In [1]:
import json
import os
import subprocess

import pandas as pd
from IPython.display import display

ROOT = "/opt/ismiy-practice7/rootfs"
LAB = ROOT + "/opt/lab"

def linux(script, timeout=120, show=True):
    if os.name == "nt":
        command = ["wsl", "-d", "Ubuntu", "-u", "root", "--exec", "bash", "-s"]
    else:
        command = (["sudo", "-n"] if os.geteuid() else []) + ["bash", "-s"]
    result = subprocess.run(command, input=script.replace("\r\n", "\n").encode("utf-8"),
                            capture_output=True, timeout=timeout)
    stdout = result.stdout.decode("utf-8", errors="replace")
    output = stdout + result.stderr.decode("utf-8", errors="replace")
    if show or result.returncode:
        print(output.strip())
    result.check_returncode()
    return stdout

def read_json(name):
    return json.loads(linux(f"cat '{LAB}/{name}'", show=False))

## Задание 1. Установка Snort и тестирование реакции

### 1. Зависимости

Команды выполнялись от root в отдельном окружении Ubuntu:

```bash
apt update
apt upgrade -y
apt install -y build-essential autoconf libpcap-dev libssl-dev flex bison pkg-config
apt install -y zlib1g-dev libluajit-5.1-dev libhyperscan-dev libpcre2-dev
apt install -y cmake automake libtool libfl-dev libdumbnet-dev libhwloc-dev \
  liblzma-dev libunwind-dev uuid-dev ca-certificates curl wget \
  iproute2 iputils-ping netcat-openbsd python3 openssl tcpdump
```

### 2. DAQ

```bash
cd /opt
wget -O libdaq-v3.0.29.tar.gz https://github.com/snort3/libdaq/archive/refs/tags/v3.0.29.tar.gz
tar -xf libdaq-v3.0.29.tar.gz
cd libdaq-3.0.29
./bootstrap
./configure --prefix=/usr/local --enable-shared
make -j$(nproc)
make install
ldconfig
```

### 3. Snort 3

```bash
cd /opt
wget -O snort3-3.13.2.0.tar.gz https://github.com/snort3/snort3/archive/refs/tags/3.13.2.0.tar.gz
tar -xf snort3-3.13.2.0.tar.gz
cd snort3-3.13.2.0
bash ./configure_cmake.sh --prefix=/usr/local
cd build
make -j$(nproc)
make install
ldconfig
snort -V
snort --daq-list
```

Для этой версии используются `configure_cmake.sh` и CMake.
Следующая ячейка содержит подготовку окружения и сборку; при повторном запуске использует установленный Snort.

In [2]:
INSTALL = r'''
set -euo pipefail
BASE=/opt/ismiy-practice7
ROOT=$BASE/rootfs
mkdir -p "$ROOT" "$BASE/logs"
if [ ! -f "$ROOT/etc/os-release" ]; then
    python3 - <<'PY'
import hashlib
import json
import tarfile
import urllib.request
from pathlib import Path
base = Path('/opt/ismiy-practice7')
digest = '26538842dfbfed820e62531e047bc6cfd3359a7e0a52f59c1a951ccc6ca787fe'
url = 'https://auth.docker.io/token?service=registry.docker.io&scope=repository:library/ubuntu:pull'
with urllib.request.urlopen(url) as response:
    token = json.load(response)['token']
request = urllib.request.Request(
    'https://registry-1.docker.io/v2/library/ubuntu/blobs/sha256:' + digest,
    headers={'Authorization': 'Bearer ' + token})
with urllib.request.urlopen(request) as response:
    data = response.read()
assert hashlib.sha256(data).hexdigest() == digest
archive = base / 'ubuntu-24.04-rootfs.tar.gz'
archive.write_bytes(data)
with tarfile.open(archive) as package:
    package.extractall(base / 'rootfs')
PY
fi
mkdir -p "$ROOT/dev" "$ROOT/proc" "$ROOT/sys"
mountpoint -q "$ROOT/dev" || mount --rbind /dev "$ROOT/dev"
mountpoint -q "$ROOT/proc" || mount -t proc proc "$ROOT/proc"
mountpoint -q "$ROOT/sys" || mount -t sysfs -o ro sysfs "$ROOT/sys"
cp /etc/resolv.conf "$ROOT/etc/resolv.conf"
if [ ! -x "$ROOT/usr/local/bin/snort" ]; then
    cat > "$ROOT/opt/install-snort.sh" <<'BUILD'
set -euo pipefail
export DEBIAN_FRONTEND=noninteractive
apt update
apt upgrade -y
apt install -y --no-install-recommends \
  build-essential autoconf libpcap-dev libssl-dev flex bison pkg-config \
  zlib1g-dev libluajit-5.1-dev libhyperscan-dev libpcre2-dev \
  cmake automake libtool libfl-dev libdumbnet-dev libhwloc-dev \
  liblzma-dev libunwind-dev uuid-dev ca-certificates curl wget \
  iproute2 iputils-ping netcat-openbsd python3 openssl tcpdump
cd /opt
wget -q -O libdaq-v3.0.29.tar.gz https://github.com/snort3/libdaq/archive/refs/tags/v3.0.29.tar.gz
tar -xf libdaq-v3.0.29.tar.gz
cd libdaq-3.0.29
./bootstrap
./configure --prefix=/usr/local --enable-shared
make -j"$(nproc)"
make install
ldconfig
cd /opt
wget -q -O snort3-3.13.2.0.tar.gz https://github.com/snort3/snort3/archive/refs/tags/3.13.2.0.tar.gz
tar -xf snort3-3.13.2.0.tar.gz
cd snort3-3.13.2.0
bash ./configure_cmake.sh --prefix=/usr/local
cd build
make -j"$(nproc)"
make install
ldconfig
BUILD
    CPUS=$(python3 -c 'import os; print(",".join(map(str, sorted(os.sched_getaffinity(0))[:4])))')
    taskset -c "$CPUS" chroot "$ROOT" bash /opt/install-snort.sh > "$BASE/logs/reproduce-build.log" 2>&1
fi
chroot "$ROOT" bash -lc 'cat /etc/os-release; snort -V; snort --daq-list'

'''
installation_output = linux(INSTALL, timeout=3600)

PRETTY_NAME="Ubuntu 24.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="24.04"
VERSION="24.04.5 LTS (Noble Numbat)"
VERSION_CODENAME=noble
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=noble
LOGO=ubuntu-logo

   ,,_     -*> Snort++ <*-
  o"  )~   Version 3.13.2.0
   ''''    By Martin Roesch & The Snort Team
           http://snort.org/contact#team
           Copyright (C) 2014-2026 Cisco and/or its affiliates. All rights reserved.
           Copyright (C) 1998-2013 Sourcefire, Inc., et al.
           Using DAQ version 3.0.29.0
           Using Hyperscan version 5.4.2 2024-04-19
           Using libpcap version 1.10.4 (with TPACKET_V3)
           Using LuaJIT version 2.1.1703358377
           Using LZMA version 5.4.5
           Using OpenSSL 3.0.13 30 Jan 2024
           Using PCRE2 version 10.42 2

### 4. Базовая настройка

Внутренний сервер — `192.168.7.100/24`, клиент — `192.168.7.10/24`.
Интерфейс наблюдения — `eth0`. Адрес `1.2.3.4` назначен отдельному тестовому адресу стенда.

В `/usr/local/etc/snort/snort.lua`:

```lua
HOME_NET = '192.168.7.0/24'
EXTERNAL_NET = '!$HOME_NET'
include 'snort_defaults.lua'

ips = {
    variables = default_variables,
    include = '/usr/local/etc/snort/rules/local.rules'
}
```

Файл `/usr/local/etc/snort/rules/local.rules`:

```snort
alert tcp any any -> $HOME_NET 80 (msg:"PRACTICE_TEST: HTTP Traffic Detected"; flow:to_server; classtype:policy-violation; sid:1000001; rev:1;)
```

### 5. Запуск

```bash
ip a
snort -c /usr/local/etc/snort/snort.lua -T
snort -c /usr/local/etc/snort/snort.lua -i eth0 -A alert_fast -k none
```

`alert_fast` выводит события в консоль. `-k none` исключает влияние checksum offload виртуальных интерфейсов.

In [3]:
SERVICES_SOURCE = r'''
import http.server
import json
import signal
import socketserver
import ssl
import threading


class HTTP(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        body = json.dumps({'path': self.path, 'status': 'lab'}, ensure_ascii=False).encode()
        self.send_response(200)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, fmt, *args):
        print('HTTP', self.client_address, fmt % args, flush=True)


class FTP(socketserver.StreamRequestHandler):
    def handle(self):
        self.wfile.write(b'220 Practice FTP control server\r\n')
        for raw in self.rfile:
            cmd = raw.decode(errors='replace').strip().split(' ', 1)[0].upper()
            response = {'USER': b'331 Password required\r\n',
                        'PASS': b'230 Logged in\r\n',
                        'QUIT': b'221 Goodbye\r\n'}.get(cmd, b'200 OK\r\n')
            self.wfile.write(response)
            if cmd == 'QUIT':
                return


class SSH(socketserver.BaseRequestHandler):
    def handle(self):
        self.request.sendall(b'SSH-2.0-Practice7\r\n')


class TCP(socketserver.ThreadingTCPServer):
    allow_reuse_address = True
    daemon_threads = True


servers = [http.server.ThreadingHTTPServer(('0.0.0.0', 80), HTTP),
           http.server.ThreadingHTTPServer(('0.0.0.0', 443), HTTP),
           TCP(('0.0.0.0', 21), FTP), TCP(('0.0.0.0', 22), SSH)]
context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
context.load_cert_chain('/opt/lab/server.crt', '/opt/lab/server.key')
servers[1].socket = context.wrap_socket(servers[1].socket, server_side=True)
for server in servers:
    threading.Thread(target=server.serve_forever, daemon=True).start()
print('READY: HTTP 80; HTTPS 443; FTP control 21; SSH banner 22', flush=True)
signal.pause()

'''

CONFIGURE_SOURCE = r'''
from pathlib import Path
import json
import subprocess

ROOT = Path('/opt/ismiy-practice7/rootfs')
LAB = ROOT / 'opt/lab'
LAB.mkdir(parents=True, exist_ok=True)
RULES = ROOT / 'usr/local/etc/snort/rules'
RULES.mkdir(parents=True, exist_ok=True)
HTTP_RULE = 'alert tcp any any -> $HOME_NET 80 (msg:"PRACTICE_TEST: HTTP Traffic Detected"; flow:to_server; classtype:policy-violation; sid:1000001; rev:1;)'
SSH_RULE = 'alert tcp $HOME_NET any -> $MALICIOUS_SERVER 22 (msg:"CRITICAL: Attempted SSH connection to known malicious server"; flow:to_server; sid:2000001; rev:1;)'
INTERPRET_RULES = [
    'alert udp $EXTERNAL_NET any -> $HOME_NET 53 (msg:"DNS Query Attempt for known MALWARE Domain"; content:"|03|abc|03|xyz|03|com|00|"; sid:2000005; rev:1; reference:url,virustotal.com/abc.xyz.com;)',
    'log tcp $HOME_NET 80 -> $EXTERNAL_NET any (msg:"Internal user accessed web server on non-standard port"; flow:to_server,established; sid:2000006;)',
    'drop icmp any any -> $HOME_NET any (msg:"Block ALL incoming ICMP requests"; sid:2000007;)'
]
for name, contents in [('local.rules', HTTP_RULE), ('ssh.rules', SSH_RULE),
                       ('interpret.rules', '\n'.join(INTERPRET_RULES))]:
    (RULES / name).write_text(contents + '\n')

sample = (ROOT / 'opt/snort3-3.13.2.0/lua/snort.lua').read_text()
sample = sample.replace("HOME_NET = 'any'", "HOME_NET = '192.168.7.0/24'", 1)
sample = sample.replace("EXTERNAL_NET = 'any'", "EXTERNAL_NET = '!$HOME_NET'", 1)
sample = sample.replace("include 'snort_defaults.lua'", "MALICIOUS_SERVER = '1.2.3.4'\ninclude 'snort_defaults.lua'\ndefault_variables.nets.MALICIOUS_SERVER = MALICIOUS_SERVER", 1)
config_dir = ROOT / 'usr/local/etc/snort'
for name, rule in [('snort.lua', 'local.rules'), ('snort-ssh.lua', 'ssh.rules'),
                   ('snort-interpret.lua', 'interpret.rules')]:
    (config_dir / name).write_text(sample + f"\nips.include = '/usr/local/etc/snort/rules/{rule}'\n")

(LAB / 'lab_services.py').write_text(SERVICES_SOURCE)
subprocess.run(['chroot', str(ROOT), 'openssl', 'req', '-x509', '-newkey', 'rsa:2048',
                '-keyout', '/opt/lab/server.key', '-out', '/opt/lab/server.crt', '-sha256',
                '-days', '3', '-nodes', '-subj', '/CN=practice7.local'], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for ns in ['ismiy-p7-client', 'ismiy-p7-sensor']:
    result = subprocess.run(['ip', 'netns', 'list'], capture_output=True, text=True, check=True)
    if ns not in result.stdout:
        subprocess.run(['ip', 'netns', 'add', ns], check=True)
exists = subprocess.run(['ip', '-n', 'ismiy-p7-client', 'link', 'show', 'eth0'],
                        capture_output=True).returncode == 0
if not exists:
    commands = [
        ['ip', 'link', 'add', 'p7-client', 'type', 'veth', 'peer', 'name', 'p7-sensor'],
        ['ip', 'link', 'set', 'p7-client', 'netns', 'ismiy-p7-client'],
        ['ip', 'link', 'set', 'p7-sensor', 'netns', 'ismiy-p7-sensor'],
        ['ip', '-n', 'ismiy-p7-client', 'link', 'set', 'p7-client', 'name', 'eth0'],
        ['ip', '-n', 'ismiy-p7-sensor', 'link', 'set', 'p7-sensor', 'name', 'eth0'],
        ['ip', '-n', 'ismiy-p7-client', 'addr', 'add', '192.168.7.10/24', 'dev', 'eth0'],
        ['ip', '-n', 'ismiy-p7-client', 'addr', 'add', '198.51.100.10/32', 'dev', 'eth0'],
        ['ip', '-n', 'ismiy-p7-sensor', 'addr', 'add', '192.168.7.100/24', 'dev', 'eth0'],
        ['ip', '-n', 'ismiy-p7-sensor', 'addr', 'add', '1.2.3.4/32', 'dev', 'eth0'],
    ]
    for command in commands:
        subprocess.run(command, check=True)
for ns in ['ismiy-p7-client', 'ismiy-p7-sensor']:
    for dev in ['lo', 'eth0']:
        subprocess.run(['ip', '-n', ns, 'link', 'set', dev, 'up'], check=True)
subprocess.run(['ip', '-n', 'ismiy-p7-client', 'route', 'replace', '1.2.3.4/32', 'dev', 'eth0'], check=True)
subprocess.run(['ip', '-n', 'ismiy-p7-sensor', 'route', 'replace', '198.51.100.10/32', 'dev', 'eth0'], check=True)

environment = {
    'sensor_interface': subprocess.check_output(['ip', '-n', 'ismiy-p7-sensor', 'address', 'show', 'dev', 'eth0'], text=True),
    'client_interface': subprocess.check_output(['ip', '-n', 'ismiy-p7-client', 'address', 'show', 'dev', 'eth0'], text=True),
    'uname': subprocess.check_output(['uname', '-a'], text=True),
    'os_release': (ROOT / 'etc/os-release').read_text(),
}
(LAB / 'environment.json').write_text(json.dumps(environment, indent=2))

results = {}
for config in ['snort.lua', 'snort-ssh.lua', 'snort-interpret.lua']:
    result = subprocess.run(['chroot', str(ROOT), 'snort', '-c',
                              '/usr/local/etc/snort/' + config, '-T'],
                             capture_output=True, text=True)
    (LAB / (config + '.validation.log')).write_text(result.stdout + result.stderr)
    results[config] = {'returncode': result.returncode, 'output': result.stdout + result.stderr}
    print(config, 'exit', result.returncode, flush=True)
    if result.returncode != 0:
        print(result.stdout + result.stderr)
    assert result.returncode == 0
(LAB / 'validation.json').write_text(json.dumps(results, indent=2))
print('HOME_NET=192.168.7.0/24 EXTERNAL_NET=!$HOME_NET MALICIOUS_SERVER=1.2.3.4 INTERFACE=eth0')

'''

TEST_SOURCE = r'''
from pathlib import Path
import json
import os
import re
import signal
import subprocess
import time

ROOT = '/opt/ismiy-practice7/rootfs'
LAB = Path(ROOT + '/opt/lab')
RESULTS = LAB / 'results'
RESULTS.mkdir(exist_ok=True)
CLIENT = 'ismiy-p7-client'
SENSOR = 'ismiy-p7-sensor'


def command(ns, args):
    prefix = ['ip', 'netns', 'exec', ns] if ns else []
    return prefix + ['chroot', ROOT] + args


def start(ns, args, path):
    stream = path.open('w')
    process = subprocess.Popen(command(ns, args), stdout=stream, stderr=subprocess.STDOUT)
    return process, stream


def finish(process, stream):
    if process.poll() is None:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=12)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)
    stream.close()


def wait_marker(process, path, marker, timeout=20):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        text = path.read_text(errors='replace') if path.exists() else ''
        if marker in text:
            return
        if process.poll() is not None:
            raise RuntimeError(f'Process stopped ({process.returncode}): {text}')
        time.sleep(0.15)
    raise RuntimeError(f'No readiness marker {marker}: {path.read_text(errors="replace")}')


server_log = LAB / 'services.log'
server, server_stream = start(SENSOR, ['python3', '-u', '/opt/lab/lab_services.py'], server_log)
wait_marker(server, server_log, 'READY:')
records = []


def test(name, args, *, config='snort.lua', sid=1000001, expect=True,
         sensor_ns=SENSOR, client_ns=CLIENT):
    directory = RESULTS / name
    directory.mkdir(exist_ok=True)
    rel = '/opt/lab/results/' + name
    snort_log = directory / 'snort.log'
    tcpdump_log = directory / 'tcpdump.log'
    sensor, sensor_stream = start(sensor_ns, ['stdbuf', '-oL', '-eL', 'snort', '-c',
        '/usr/local/etc/snort/' + config, '-i', 'eth0', '-A', 'alert_fast',
        '-k', 'none', '-l', rel], snort_log)
    capture, capture_stream = start(sensor_ns, ['tcpdump', '-i', 'eth0', '-nn', '-U',
        '-w', rel + '/traffic.pcap'], tcpdump_log)
    try:
        wait_marker(sensor, snort_log, 'Commencing packet processing')
        wait_marker(capture, tcpdump_log, 'listening on eth0')
        result = subprocess.run(command(client_ns, args), capture_output=True, text=True, timeout=30)
        time.sleep(0.7)
    finally:
        finish(sensor, sensor_stream)
        finish(capture, capture_stream)
    text = snort_log.read_text(errors='replace')
    alerts = [line for line in text.splitlines() if f'[1:{sid}:1]' in line]
    client_text = result.stdout + result.stderr
    (directory / 'client.log').write_text(client_text)
    stats = subprocess.run(command(sensor_ns, ['tcpdump', '-nn', '-r', rel + '/traffic.pcap']),
                           capture_output=True, text=True, check=True)
    packet_lines = stats.stdout.splitlines()
    record = dict(name=name, command=args, client_returncode=result.returncode,
                  client_output=client_text, sid=sid, alert_count=len(alerts),
                  alerts=alerts, expected_alert=expect,
                  passed=bool(alerts) == expect and result.returncode == 0,
                  captured_packets=len(packet_lines),
                  captured_summary=packet_lines[:6], snort_returncode=sensor.returncode)
    records.append(record)
    print(name, 'exit', result.returncode, 'packets', len(packet_lines),
          'alerts', len(alerts), 'PASS' if record['passed'] else 'FAIL', flush=True)
    if not record['passed']:
        print(client_text, flush=True)
        print('\n'.join(text.splitlines()[-35:]), flush=True)
    (LAB / 'results.json').write_text(json.dumps(records, ensure_ascii=False, indent=2))


try:
    test('icmp', ['ping', '-c', '2', '-W', '1', '192.168.7.100'], expect=False)
    test('http', ['curl', '--noproxy', '*', '-sS', '--max-time', '10', 'http://192.168.7.100/'])
    test('https_local', ['curl', '--noproxy', '*', '-ksS', '--max-time', '10',
                         'https://192.168.7.100/'], expect=False)
    test('https_google', ['curl', '-sS', '--max-time', '20', '-o', '/dev/null',
                          '-w', 'HTTP %{http_code}; bytes %{size_download}\\n',
                          'https://www.google.com'], expect=False, sensor_ns=None, client_ns=None)
    test('ftp', ['bash', '-c', "printf 'USER anonymous\\r\\nPASS lab\\r\\nQUIT\\r\\n' | nc -w 2 192.168.7.100 21"], expect=False)
    test('sqli_http', ['curl', '--noproxy', '*', '-sS', '--max-time', '10', '--get',
                       '--data-urlencode', "username=admin' OR 1=1 --", 'http://192.168.7.100/login'])
    test('ssh_target', ['nc', '-w', '2', '1.2.3.4', '22'], config='snort-ssh.lua', sid=2000001)
    test('ssh_other_target', ['nc', '-w', '2', '192.168.7.100', '22'],
         config='snort-ssh.lua', sid=2000001, expect=False)
    test('ssh_other_port', ['curl', '--noproxy', '*', '-sS', '--max-time', '10', 'http://1.2.3.4/'],
         config='snort-ssh.lua', sid=2000001, expect=False)
    test('ssh_external_source', ['nc', '-s', '198.51.100.10', '-w', '2', '1.2.3.4', '22'],
         config='snort-ssh.lua', sid=2000001, expect=False)
finally:
    finish(server, server_stream)

assert all(record['passed'] for record in records), 'Some live cases failed'
assert all(record['captured_packets'] > 0 for record in records), 'Capture must observe every test'
print('All live traffic cases passed.', flush=True)

'''

def run_live_tests():
    preamble = f"""set -euo pipefail
ROOT={ROOT}
mkdir -p "$ROOT/dev" "$ROOT/proc" "$ROOT/sys"
mountpoint -q "$ROOT/dev" || mount --rbind /dev "$ROOT/dev"
mountpoint -q "$ROOT/proc" || mount -t proc proc "$ROOT/proc"
mountpoint -q "$ROOT/sys" || mount -t sysfs -o ro sysfs "$ROOT/sys"
cp /etc/resolv.conf "$ROOT/etc/resolv.conf"
trap 'ip netns del ismiy-p7-client 2>/dev/null || true; ip netns del ismiy-p7-sensor 2>/dev/null || true' EXIT
"""
    source = ("SERVICES_SOURCE = " + repr(SERVICES_SOURCE) + "\n" +
              CONFIGURE_SOURCE + "\n" + TEST_SOURCE)
    linux(preamble + "python3 - <<'PRACTICE7'\n" + source + "\nPRACTICE7\n", timeout=300)

In [4]:
run_live_tests()

snort.lua exit 0
snort-ssh.lua exit 0
snort-interpret.lua exit 0
HOME_NET=192.168.7.0/24 EXTERNAL_NET=!$HOME_NET MALICIOUS_SERVER=1.2.3.4 INTERFACE=eth0
icmp exit 0 packets 13 alerts 0 PASS
http exit 0 packets 13 alerts 7 PASS
https_local exit 0 packets 15 alerts 0 PASS
https_google exit 0 packets 56 alerts 0 PASS
ftp exit 0 packets 17 alerts 0 PASS
sqli_http exit 0 packets 13 alerts 8 PASS
ssh_target exit 0 packets 10 alerts 4 PASS
ssh_other_target exit 0 packets 8 alerts 0 PASS
ssh_other_port exit 0 packets 13 alerts 0 PASS
ssh_external_source exit 0 packets 10 alerts 0 PASS
All live traffic cases passed.


In [5]:
environment = read_json("environment.json")
print(environment["sensor_interface"])
validation = read_json("validation.json")
for config, info in validation.items():
    lines = [line for line in info["output"].splitlines()
             if "successfully validated" in line or "rules loaded" in line]
    print(config, "exit", info["returncode"])
    print("\n".join(lines))

5: eth0@if6: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc noqueue state UP group default qlen 1000
    link/ether 82:cc:6d:06:47:0a brd ff:ff:ff:ff:ff:ff link-netns ismiy-p7-client
    inet 192.168.7.100/24 scope global eth0
       valid_lft forever preferred_lft forever
    inet 1.2.3.4/32 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::80cc:6dff:fe06:470a/64 scope link tentative 
       valid_lft forever preferred_lft forever



snort.lua exit 0
       total rules loaded: 220
Snort successfully validated the configuration (with 0 warnings).
snort-ssh.lua exit 0
       total rules loaded: 220
Snort successfully validated the configuration (with 0 warnings).
snort-interpret.lua exit 0
       total rules loaded: 222
Snort successfully validated the configuration (with 0 warnings).


### 6. Имитация трафика

Для каждого опыта Snort запускался отдельно. Дополнительно сохранялся PCAP,
чтобы подтвердить поступление пакетов на интерфейс.

| Трафик | Команда | Ожидаемый результат для SID 1000001 |
| :--- | :--- | :--- |
| ICMP | `ping -c 2 192.168.7.100` | Нет: правило только для TCP/80. |
| HTTP | `curl http://192.168.7.100/` | Есть: запрос к внутреннему серверу на TCP/80. |
| HTTPS | `curl https://www.google.com` | Нет: TCP/443. |
| FTP | `printf 'USER anonymous\r\nPASS lab\r\nQUIT\r\n' \| nc -w 2 192.168.7.100 21` | Нет: управляющее соединение TCP/21. |
| HTTP с SQL-подобным параметром | `curl --get --data-urlencode "username=admin' OR 1=1 --" http://192.168.7.100/login` | Есть: TCP/80, без распознавания SQLi. |

HTTPS дополнительно проверен на внутреннем TLS-сервере на порту 443;
`-k` использован только для его самоподписанного сертификата.

In [6]:
records = read_json("results.json")
by_name = {item["name"]: item for item in records}
names = ["icmp", "http", "https_google", "ftp", "sqli_http", "https_local"]
labels = ["ICMP", "HTTP", "HTTPS Google", "FTP/21", "HTTP с SQL-параметром", "HTTPS стенда"]
display(pd.DataFrame([
    {"Трафик": label, "Пакетов в PCAP": by_name[name]["captured_packets"],
     "Алертов 1000001": by_name[name]["alert_count"],
     "Код команды": by_name[name]["client_returncode"],
     "Результат": "верно" if by_name[name]["passed"] else "ошибка"}
    for name, label in zip(names, labels)
]))
for name in ["http", "https_google", "ftp", "sqli_http"]:
    print(name + ":", by_name[name]["client_output"].strip())
    print("\n".join(by_name[name]["alerts"][:2]))

,Трафик,Пакетов в PCAP,Алертов 1000001,Код команды,Результат
0,ICMP,13,0,0,верно
1,HTTP,13,7,0,верно
2,HTTPS Google,56,0,0,верно
3,FTP/21,17,0,0,верно
4,HTTP с SQL-параметром,13,8,0,верно
5,HTTPS стенда,15,0,0,верно


http: {"path": "/", "status": "lab"}
11/14-10:32:38.476531 [**] [1:1000001:1] "PRACTICE_TEST: HTTP Traffic Detected" [**] [Classification: Potential Corporate Privacy Violation] [Priority: 1] {TCP} 192.168.7.10:46116 -> 192.168.7.100:80
11/14-10:32:38.476643 [**] [1:1000001:1] "PRACTICE_TEST: HTTP Traffic Detected" [**] [Classification: Potential Corporate Privacy Violation] [Priority: 1] {TCP} 192.168.7.10:46116 -> 192.168.7.100:80
https_google: HTTP 200; bytes 89050

ftp: 220 Practice FTP control server
331 Password required
230 Logged in
221 Goodbye

sqli_http: {"path": "/login?username=admin%27+OR+1%3d1+--", "status": "lab"}
11/14-10:33:13.611779 [**] [1:1000001:1] "PRACTICE_TEST: HTTP Traffic Detected" [**] [Classification: Potential Corporate Privacy Violation] [Priority: 1] {TCP} 192.168.7.10:33732 -> 192.168.7.100:80
11/14-10:33:13.611837 [**] [1:1000001:1] "PRACTICE_TEST: HTTP Traffic Detected" [**] [Classification: Potential Corporate Privacy Violation] [Priority: 1] {TCP} 19

HTTP и HTTP-запрос с SQL-подобной строкой соответствуют правилу по направлению и порту.
Это правило не анализирует SQL-синтаксис. ICMP, HTTPS и FTP не соответствуют TCP/80.
Число алертов относится к событиям Snort, а не к числу уникальных запросов.

## Задание 2. Чтение и интерпретация правил

### Правило 1

```snort
alert udp $EXTERNAL_NET any -> $HOME_NET 53 (msg:"DNS Query Attempt for known MALWARE Domain"; content:"|03|abc|03|xyz|03|com|00|"; sid:2000005; rev:1; reference:url,virustotal.com/abc.xyz.com;)
```

- **Действие:** `alert` — сформировать оповещение.
- **Протокол и порты:** UDP, любой исходный порт внешнего узла → порт 53 внутреннего узла.
- **Сигнатура:** байты `03 abc 03 xyz 03 com 00`, то есть DNS-представление имени `abc.xyz.com`.
  Опция `reference` задает справочную ссылку.

### Правило 2

```snort
log tcp $HOME_NET 80 -> $EXTERNAL_NET any (msg:"Internal user accessed web server on non-standard port"; flow:to_server,established; sid:2000006;)
```

- **Действие:** `log` — записать пакет в настроенный журнал пакетов.
- **Протокол и порты:** TCP, внутренний узел с **исходным портом 80** → любой порт внешнего узла.
- **Условие:** установленное соединение, направление к серверу; `content` или другой шаблон содержимого отсутствует.
  Несмотря на `msg`, нестандартный порт назначения правилом не задан.

### Правило 3

```snort
drop icmp any any -> $HOME_NET any (msg:"Block ALL incoming ICMP requests"; sid:2000007;)
```

- **Действие:** `drop` — отбросить пакет в режиме IPS/inline.
- **Протокол и порты:** ICMP от любого источника во внутреннюю сеть; у ICMP нет TCP/UDP-портов.
- **Условие:** любой ICMP-пакет к HOME_NET, без фильтра по типу и содержимому.
  Правило охватывает и запросы, и ответы. В пассивном IDS-режиме фактической блокировки нет.

Все три правила прошли проверку загрузки Snort (`snort-interpret.lua -T`).

## Задание 3. Создание базового правила

Переменная адреса и ее регистрация в конфигурации:

```lua
HOME_NET = '192.168.7.0/24'
EXTERNAL_NET = '!$HOME_NET'
MALICIOUS_SERVER = '1.2.3.4'
include 'snort_defaults.lua'
default_variables.nets.MALICIOUS_SERVER = MALICIOUS_SERVER

ips = {
    variables = default_variables,
    include = '/usr/local/etc/snort/rules/ssh.rules'
}
```

Файл `/usr/local/etc/snort/rules/ssh.rules`:

```snort
alert tcp $HOME_NET any -> $MALICIOUS_SERVER 22 (msg:"CRITICAL: Attempted SSH connection to known malicious server"; flow:to_server; sid:2000001; rev:1;)
```

`alert` формирует оповещение при исходящем TCP-трафике внутреннего хоста к `1.2.3.4:22`.
`flow:to_server` задает направление, `sid` — идентификатор, `rev` — редакцию.
Правило фиксирует попытку соединения, не подтверждая успешную аутентификацию.

Проверка:

```bash
snort -c /usr/local/etc/snort/snort-ssh.lua -T
snort -c /usr/local/etc/snort/snort-ssh.lua -i eth0 -A alert_fast -k none
nc -w 2 1.2.3.4 22
```

In [7]:
ssh_names = ["ssh_target", "ssh_other_target", "ssh_other_port", "ssh_external_source"]
ssh_labels = ["HOME_NET → 1.2.3.4:22", "HOME_NET → 192.168.7.100:22",
              "HOME_NET → 1.2.3.4:80", "Внешний источник → 1.2.3.4:22"]
display(pd.DataFrame([
    {"Проверка": label, "Пакетов в PCAP": by_name[name]["captured_packets"],
     "Алертов 2000001": by_name[name]["alert_count"],
     "Результат": "верно" if by_name[name]["passed"] else "ошибка"}
    for name, label in zip(ssh_names, ssh_labels)
]))
print("\n".join(by_name["ssh_target"]["alerts"][:3]))

,Проверка,Пакетов в PCAP,Алертов 2000001,Результат
0,HOME_NET → 1.2.3.4:22,10,4,верно
1,HOME_NET → 192.168.7.100:22,8,0,верно
2,HOME_NET → 1.2.3.4:80,13,0,верно
3,Внешний источник → 1.2.3.4:22,10,0,верно


11/14-10:33:26.622134 [**] [1:2000001:1] "CRITICAL: Attempted SSH connection to known malicious server" [**] [Priority: 0] {TCP} 192.168.7.10:34356 -> 1.2.3.4:22
11/14-10:33:26.622214 [**] [1:2000001:1] "CRITICAL: Attempted SSH connection to known malicious server" [**] [Priority: 0] {TCP} 192.168.7.10:34356 -> 1.2.3.4:22
11/14-10:33:26.622938 [**] [1:2000001:1] "CRITICAL: Attempted SSH connection to known malicious server" [**] [Priority: 0] {TCP} 192.168.7.10:34356 -> 1.2.3.4:22


Оповещение SID 2000001 появилось для внутреннего источника и `1.2.3.4:22`.
При другом адресе, порте и внешнем источнике срабатываний этого правила нет.

## Контрольные вопросы

**1. Какой ключевой недостаток выявился при анализе HTTPS-трафика с помощью Snort без дополнительных модулей?**

TLS скрывает содержимое HTTP: URI, заголовки и тело запроса недоступны для обычных контентных сигнатур.
Snort может анализировать доступные метаданные TLS и соединения.
Для проверки содержимого нужна контролируемая расшифровка.
Тестовое правило дополнительно ограничено портом 80 и не охватывает HTTPS/443.

**2. Для чего необходима библиотека DAQ при установке Snort из исходных кодов?**

DAQ предоставляет Snort интерфейс получения пакетов и работы с источником трафика.
Модули DAQ обеспечивают захват с интерфейса, чтение PCAP и работу в подходящем inline-режиме.

**3. Как вы определили переменную HOME_NET в файле snort.lua?**

По адресу интерфейса сервера `192.168.7.100/24`, проверенному через `ip a`.
Маска `/24` соответствует `255.255.255.0`, поэтому сеть — `192.168.7.0/24`.
В `snort.lua` задано `HOME_NET = '192.168.7.0/24'`, а `EXTERNAL_NET = '!$HOME_NET'`.